# Oppgave 1: Hvor mange mennesker bor nært kjøpesentrene? (10 poeng)

I denne oppgaven fortsetter vi fra den forrige. Du skal gjøre en romlig kobling for å relatere data fra et befolkningsrutenett-datasett til bufferlaget opprettet i forrige øving for å finne ut hvor mange mennesker som bor i alle befolkningsrutenettceller som er innenfor 1,5 km avstand fra hvert kjøpesenter.

Bruk det samme befolkningsrutenettet som ble brukt i forelesningen og bufferlaget (shopping_centres.geojson) du lagde i forrige øving. Ha begge datafilene i samme mappe som denne notebooken, og lever både begge datafilene og notebooken til slutt.

Du kan gjerne lage flere kodeceller for å løse oppgaven, bare husk å lag de før test-cellene, og ikke slett test-cellene!

Husk å kommentere koden din underveis.

In [1]:
# Imports
import pathlib
import geopandas as gpd

## 1a) Laste inn rutenett

Bruk det samme befolkningsrutenettet som ble brukt i forelesningen. Last det inn i en `GeoDataFrame` som har navnet `rutenett`. Reprojiser rutenettet til CRS med EPSG-kode `25832`.

In [2]:
# SKRIV DIN KODE HER OG FJERN LINJEN "raise NotImplementedError()"

NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_MAPPE = NOTEBOOK_PATH / "data"     

rutenett = gpd.read_file(
    DATA_MAPPE / "ssb_rutenett" / "befolkning_250m_2023_oslo.shp"   # Leser inn data-filen til befolkningsrutenettet
    )                                                               
rutenett = rutenett.to_crs("EPSG:25832")                            # Reprojisering av rutenett til riktig CRS (EPSG:25832)
rutenett.head()

,SSBID0250M,pop_tot,geometry
0,22522506661750,2,"POLYGON ((586853.053 6654456.688, 586604.245 6..."
1,22525006661500,3,"POLYGON ((587124.5 6654230.519, 586875.692 665..."
2,22527506661750,2,"POLYGON ((587350.669 6654501.967, 587101.861 6..."
3,22530006662000,7,"POLYGON ((587576.838 6654773.416, 587328.029 6..."
4,22532506662000,1,"POLYGON ((587825.647 6654796.056, 587576.838 6..."


In [4]:
# TEST-CELLE
## Ikke slett denne cellen, og nye kodeceller du lager til denne deloppgaven skal komme før denne!
import geopandas
import pyproj
assert isinstance(rutenett, geopandas.GeoDataFrame), "Finner ikke GeoDataFramen 'rutenett'"
assert rutenett.crs == pyproj.CRS("EPSG:25832"), "Sjekk at GeoDataFramen har riktig CRS"


## 1b) Last inn bufferlag

Last inn bufferne du lagde i forrige øving inn i en `GeoDataFrame` som heter `buffer_lag`. 

Forsikre deg om at alle kjøpesentrene har en buffer, hvis ikke må du gjøre geokodingen fra Øving 4 om igjen, til du har en buffer per kjøpesenter.

Lag en `assert` som sjekker hvorvidt begge geodataframene har samme CRS.

In [5]:
# SKRIV DIN KODE HER OG FJERN LINJEN "raise NotImplementedError()"

buffer_lag = gpd.read_file(DATA_MAPPE / "shopping_centres.geojson")         # Leser inn filen med bufferen rundt kjøpesenterene

In [6]:
assert buffer_lag.crs == rutenett.crs, "GeoDataFrame-ene har ikke samme CRS."   # Sjekker at begge GeoDataFrame-ene har samme CRS

In [7]:
# TEST-CELLE
## Ikke slett denne cellen, og nye kodeceller du lager til denne deloppgaven skal komme før denne!
assert isinstance(buffer_lag, geopandas.GeoDataFrame), "Finner ikke GeoDataFramen 'buffer_lag'"


## 1c) Utfør en romlig kobling mellom `rutenett` og `buffer_lag`

Slå sammen bufferlagets `id`-kolonne (og andre, hvis du vil) med befolkningsrutenett-dataframen, for alle befolkningsrutenettceller som er innenfor bufferområdet til hvert kjøpesenter. [Bruk en sammenkoblingstype som beholder kun rader fra begge inndataframene der den geometriske predikaten er sann](https://geopandas.org/en/stable/gallery/spatial_joins.html#Types-of-spatial-joins).

Lagre resultatet i en variabel med navnet `sammenkoblet_data`.

In [8]:
# SKRIV DIN KODE HER OG FJERN LINJEN "raise NotImplementedError()"

sammenkoblet_data = gpd.sjoin(                          # Sammenkobler data fra befolknignsrutenett og bufferlaget med inner join og predikatet 'within'
    rutenett,
    buffer_lag[["id", "navn", "geometry"]],
    how="inner",
    predicate="within"
)               

sammenkoblet_data.head()

,SSBID0250M,pop_tot,geometry,index_right,id,navn
663,22612506649500,6,"POLYGON ((596919.28 6643079.205, 596670.452 66...",0,1,Oslo City
684,22615006649000,4,"POLYGON ((597213.342 6642604.163, 596964.513 6...",0,1,Oslo City
685,22615006649250,2,"POLYGON ((597190.726 6642852.992, 596941.897 6...",0,1,Oslo City
686,22615006649500,1,"POLYGON ((597168.109 6643101.822, 596919.28 66...",0,1,Oslo City
687,22615006649750,46,"POLYGON ((597145.493 6643350.651, 596896.664 6...",0,1,Oslo City


In [9]:
# TEST-CELLE
## Ikke slett denne cellen, og nye kodeceller du lager til denne deloppgaven skal komme før denne!
assert isinstance(sammenkoblet_data, geopandas.GeoDataFrame), "Finner ikke GeoDataFramen 'sammenkoblet_data'"

## 1d) Regn ut den totale befolkningen rundt kjøpesentrene

Grupper den resulterende (sammenkoblede) dataframen etter kjøpesenternavn, og beregn summen (`sum()`) av befolkningen som bor innenfor en radius på 1,5 km rundt dem.

Skriv ut resultatene, for eksempel i formen "12345 mennesker bor innen 1,5 km fra Oslo City".

In [10]:
# SKRIV DIN KODE HER OG FJERN LINJEN "raise NotImplementedError()"

befolkning_bydel = sammenkoblet_data.groupby("navn")["pop_tot"].sum()    # Summerer befolkningen i hver bydel

for navn, pop in befolkning_bydel.items():
    print(f"{pop} mennesker bor innen 1.5 km fra {navn}")                # Skriver ut reslutatene

22789 mennesker bor innen 1.5 km fra Bryn Senter
25771 mennesker bor innen 1.5 km fra Lambertseter Senter
22867 mennesker bor innen 1.5 km fra Linderud Senter
22759 mennesker bor innen 1.5 km fra Manglerud Senter
48050 mennesker bor innen 1.5 km fra Oslo City
44401 mennesker bor innen 1.5 km fra Stor Storsenter
20920 mennesker bor innen 1.5 km fra Tveita Senter


In [11]:
# TEST-CELLE
## Ikke slett denne cellen, og nye kodeceller du lager til denne deloppgaven skal komme før denne!


## 1e) Refleksjon

Godt jobbet! Du er nesten ferdig med denne ukens øvelse. Vennligst svar raskt på følgende korte spørsmål:

* Hvor utfordrende syntes du denne og forrige øvingsoppgave var (på en skala fra 1-5), og hvorfor?
* Hva var lett?
* Hva var vanskelig?

Legg til svarene dine i Markdown-cellen nedenfor:

Jeg synes både øving 4 og denne er en 3 av 5. Jeg synes ikke selve programmeringen var så veldig vanskelig, men jeg ble utfordret med tanke på de bakenforliggende konseptene i geokoding og join. Alt i alt fine øvinger som får med det meste fra forelesninger. 